In [ ]:
# =============================================================
# 09_task_E_causal_ml.py
# Task E — Causal Machine Learning
# ------------------------------------------------------------
# Benchmark task: estimate the Average Treatment Effect (ATE)
# and Conditional Average Treatment Effect (CATE) of
# government education expenditure (W_it) on lower-secondary
# completion rate (Y_it), controlling for confounders X_it.
#
# Causal model (§2, §5.5)
# ───────────────────────
#   Y_it = τ(X_it) · W_it + g(X_it) + ε_it
#
#   Y  = sec_completion          (outcome)
#   W  = edu_spend_gdp           (treatment — education spend % GDP)
#   X  = {gdp_pc_ppp, gov_effect, gini, gpi_sec,
#          sec_enrol_gross, tert_enrol_gross,
#          work_age_share, voice_account}
#   A  = gpi_sec                 (protected attribute, subgroup CATEs)
#
# Treatment variable note
# ───────────────────────
# edu_spend_gdp is a flow variable (annual spend). We use
# the contemporaneous value W_it (not lagged) as the treatment,
# consistent with the within-year fiscal policy channel.
# Confounders X_it are lagged one year (X_{i,t-1}) to avoid
# post-treatment conditioning.
#
# Estimators
# ──────────
# 1. Naïve OLS (no causal adjustment — biased baseline)
# 2. Double Machine Learning — Linear (LinearDML)
#    Nuisance: Ridge regression for Y~X and W~X residuals
# 3. Double Machine Learning — Causal Forest (CausalForestDML)
#    Heterogeneous CATE τ(X_it)
# 4. Panel Fixed Effects (within-country demeaning)
#    Classical econometric benchmark
#
# Evaluation
# ──────────
# - ATE point estimate ± 95% CI
# - CATE distribution: mean, std, IQR
# - Subgroup ATE by trajectory class (EE/AE/IR/DL)
# - Subgroup ATE by GPI group (favoured/unfavoured)
# - Robustness: coefficient stability across covariate sets
#
# Outputs
# ───────
#   outputs/results/task_E_ate.csv
#   outputs/results/task_E_cate.csv
#   outputs/tables/table_taskE.csv / .tex
#   outputs/figures/panel_12/taskE_cate_distribution.png
#   outputs/figures/panel_12/taskE_ate_comparison.png
#   outputs/figures/panel_12/taskE_subgroup_ate.png
#
# Usage
# ─────
#   python src/09_task_E_causal_ml.py
#
# Requirements
# ────────────
#   pip install econml scikit-learn pandas numpy matplotlib pyyaml
# =============================================================

from __future__ import annotations

import sys
import warnings
from pathlib import Path

try:
    _SRC = Path(__file__).resolve().parent
except NameError:
    _cwd = Path.cwd()
    _SRC = _cwd / "src" if (_cwd / "src").exists() else _cwd
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from econml.dml import LinearDML, CausalForestDML

from utils import find_project_root, load_indicators, load_pipeline, INCOME_ORDER

warnings.filterwarnings("ignore")


# ──────────────────────────────────────────────────────────────
# Setup
# ──────────────────────────────────────────────────────────────
PROJECT_ROOT = find_project_root()
IND_CFG      = load_indicators()
PIPE_CFG     = load_pipeline()

YEAR_MIN   = PIPE_CFG["years"]["min"]
YEAR_MAX   = PIPE_CFG["years"]["max"]
PERIOD_TAG = f"{YEAR_MIN}_{YEAR_MAX}"
TRAIN_END  = 2021

OUTCOME   = "sec_completion"
TREATMENT = "edu_spend_gdp"
PROTECTED = "gpi_sec"

# Confounders — lagged 1 year to avoid post-treatment conditioning
CONFOUNDERS = [
    "gdp_pc_ppp",
    "gov_effect",
    "gini",
    "gpi_sec",
    "sec_enrol_gross",
    "tert_enrol_gross",
    "work_age_share",
    "voice_account",
]

DATA_DIR   = PROJECT_ROOT / "data" / "processed"
SPLIT_DIR  = DATA_DIR / "train_test_splits"
RESULT_DIR = PROJECT_ROOT / "outputs" / "results"
TABLE_DIR  = PROJECT_ROOT / "outputs" / "tables"
FIGDIR     = PROJECT_ROOT / "outputs" / "figures" / "panel_12"

for d in [RESULT_DIR, TABLE_DIR, FIGDIR]:
    d.mkdir(parents=True, exist_ok=True)

CLASS_COLOURS = {
    "EE": "#1a7a3a", "AE": "#1a6fad",
    "IR": "#d73027", "DL": "#888888",
}

print(f"[info] Project root  : {PROJECT_ROOT}")
print(f"[info] Outcome       : {OUTCOME}")
print(f"[info] Treatment     : {TREATMENT}")
print(f"[info] Confounders   : {len(CONFOUNDERS)}")


# ──────────────────────────────────────────────────────────────
# Load data
# ──────────────────────────────────────────────────────────────
def load_data() -> tuple[pd.DataFrame, pd.DataFrame]:
    # Prefer imputed panel (Task D output); fall back to raw panel
    imp_path   = DATA_DIR / f"panel_12countries_imputed_{PERIOD_TAG}.csv"
    raw_path   = DATA_DIR / f"panel_12countries_{PERIOD_TAG}.csv"
    label_path = DATA_DIR / "trajectory_labels.csv"

    panel_path = imp_path if imp_path.exists() else raw_path
    print(f"\n[step 0] Using panel: {panel_path.name}")

    for p in [panel_path, label_path]:
        if not p.exists():
            raise FileNotFoundError(f"Required: {p}")

    panel  = pd.read_csv(panel_path)
    labels = pd.read_csv(label_path)
    print(f"  Panel  : {len(panel)} rows × {len(panel.columns)} cols")
    print(f"  Labels : {len(labels)} countries")
    return panel, labels


# ──────────────────────────────────────────────────────────────
# Build causal arrays
# ──────────────────────────────────────────────────────────────
def build_causal_arrays(
    panel: pd.DataFrame,
    labels: pd.DataFrame,
) -> dict:
    """
    Construct Y, W, X arrays with:
      - Confounders X_{i,t-1} lagged one year
      - Treatment W_it contemporaneous
      - Outcome Y_it contemporaneous
      - Country and year dummies for FE model
    Rows with any missing in Y or W are dropped.
    Missing X imputed with column median.
    """
    p = panel.copy().sort_values(["iso3c", "year"])

    # Lag confounders
    for conf in CONFOUNDERS:
        if conf in p.columns:
            p[f"{conf}_lag1"] = p.groupby("iso3c")[conf].shift(1)

    lag_cols = [f"{c}_lag1" for c in CONFOUNDERS if f"{c}_lag1" in p.columns]

    # Drop rows missing outcome or treatment
    p = p.dropna(subset=[OUTCOME, TREATMENT])
    # Drop first year (no lag available)
    p = p[p["year"] > YEAR_MIN].copy()

    # Impute X
    imp = SimpleImputer(strategy="median")
    X_raw = p[lag_cols].values.astype(float)
    X     = imp.fit_transform(X_raw)

    Y = p[OUTCOME].values.astype(float)
    W = p[TREATMENT].values.astype(float)

    # Country dummies for FE
    country_dummies = pd.get_dummies(p["iso3c"], drop_first=True).values

    # GPI group (protected attribute)
    prot_lag = "gpi_sec_lag1" if "gpi_sec_lag1" in p.columns else None
    if prot_lag:
        A_raw = p[prot_lag].values.astype(float)
        A_imp = SimpleImputer(strategy="median")
        A     = A_imp.fit_transform(A_raw.reshape(-1,1)).ravel()
    else:
        A = np.ones(len(Y))

    gpi_median = float(np.nanmedian(A))
    favoured   = A >= gpi_median

    # Merge trajectory labels
    iso_arr = p["iso3c"].values
    label_map = dict(zip(labels["iso3c"], labels["label_code"]))
    label_arr = np.array([label_map.get(iso, "??") for iso in iso_arr])

    print(f"\n[step 1] Causal arrays:")
    print(f"  n rows          : {len(Y)}")
    print(f"  W range         : [{W.min():.2f}, {W.max():.2f}] % GDP")
    print(f"  Y range         : [{Y.min():.1f}, {Y.max():.1f}] %")
    print(f"  X dimensions    : {X.shape[1]}")
    print(f"  GPI median      : {gpi_median:.3f}")
    print(f"  Favoured n      : {favoured.sum()}  "
          f"Unfavoured n: {(~favoured).sum()}")

    # Train mask (within YEAR_MIN+1 to TRAIN_END)
    yr_arr     = p["year"].values
    train_mask = yr_arr <= TRAIN_END

    return {
        "Y": Y, "W": W, "X": X, "A": A,
        "favoured": favoured,
        "label_arr": label_arr,
        "iso_arr": iso_arr,
        "yr_arr": yr_arr,
        "train_mask": train_mask,
        "country_dummies": country_dummies,
        "lag_cols": lag_cols,
        "imp": imp,
        "gpi_median": gpi_median,
        "df": p,
    }


# ──────────────────────────────────────────────────────────────
# Estimator 1 — Naïve OLS
# ──────────────────────────────────────────────────────────────
def run_naive_ols(arrays: dict) -> dict:
    """
    Regress Y on [W, X] without any causal adjustment.
    Coefficient on W is the naïve (confounded) estimate.
    """
    Y, W, X = arrays["Y"], arrays["W"], arrays["X"]
    WX      = np.column_stack([W.reshape(-1,1), X])

    sc   = StandardScaler()
    WX_s = sc.fit_transform(WX)

    lr   = LinearRegression()
    lr.fit(WX_s, Y)

    # Coefficient on W (first column after scaling)
    coef_W  = lr.coef_[0]
    # Unstandardise: multiply by std(W)/std(Y-like) → undo W scaling
    w_std   = float(np.std(W))
    coef_pp = coef_W * w_std   # pp change per 1 pp-GDP increase in spend

    return {
        "estimator": "NaiveOLS",
        "ate":       round(coef_pp, 4),
        "ate_lower": np.nan,
        "ate_upper": np.nan,
        "note": "Confounded; no causal adjustment",
    }


# ──────────────────────────────────────────────────────────────
# Estimator 2 — Panel Fixed Effects (within demeaning)
# ──────────────────────────────────────────────────────────────
def run_fixed_effects(arrays: dict) -> dict:
    """
    Within-country fixed effects via demeaning.
    Demean Y, W, X by country mean → partial out country-level confounders.
    OLS on demeaned variables gives the FE estimator.
    """
    Y, W, X = arrays["Y"], arrays["W"], arrays["X"]
    iso_arr = arrays["iso_arr"]
    df      = arrays["df"].copy()
    df["_Y"] = Y
    df["_W"] = W
    for j, col in enumerate(arrays["lag_cols"]):
        df[f"_X{j}"] = X[:, j]

    x_cols = [f"_X{j}" for j in range(X.shape[1])]

    # Demean within country
    for col in ["_Y", "_W"] + x_cols:
        df[f"{col}_dm"] = df[col] - df.groupby("iso3c")[col].transform("mean")

    dm_cols   = ["_W_dm"] + [f"{c}_dm" for c in x_cols]
    X_fe      = df[dm_cols].values
    Y_fe      = df["_Y_dm"].values

    lr = LinearRegression(fit_intercept=False)
    lr.fit(X_fe, Y_fe)

    coef_W = float(lr.coef_[0])

    # Bootstrap SE (residual bootstrap, 200 iterations)
    rng      = np.random.default_rng(42)
    y_hat    = lr.predict(X_fe)
    residuals = Y_fe - y_hat
    boot_coefs = []
    for _ in range(200):
        idx    = rng.integers(0, len(Y_fe), size=len(Y_fe))
        Y_b    = y_hat + residuals[idx]
        lr_b   = LinearRegression(fit_intercept=False)
        lr_b.fit(X_fe, Y_b)
        boot_coefs.append(lr_b.coef_[0])

    boot_arr = np.array(boot_coefs)
    ci_lo    = float(np.percentile(boot_arr, 2.5))
    ci_hi    = float(np.percentile(boot_arr, 97.5))

    return {
        "estimator": "FixedEffects",
        "ate":       round(coef_W, 4),
        "ate_lower": round(ci_lo, 4),
        "ate_upper": round(ci_hi, 4),
        "note": "Within-country FE (demeaned), bootstrap 95% CI",
    }


# ──────────────────────────────────────────────────────────────
# Estimator 3 — Double Machine Learning (Linear)
# ──────────────────────────────────────────────────────────────
def run_linear_dml(arrays: dict) -> dict:
    """
    LinearDML from econml:
      - Nuisance models: Ridge for E[Y|X] and E[W|X]
      - Final model: OLS on partialled-out residuals
      - Cross-fitting: 5-fold CV
      - Inference: asymptotic normal CI
    """
    Y, W, X = arrays["Y"], arrays["W"], arrays["X"]

    dml = LinearDML(
        model_y=Ridge(alpha=1.0),
        model_t=Ridge(alpha=1.0),
        cv=5,
        random_state=42,
        discrete_treatment=False,
    )
    dml.fit(Y, W, X=X)

    ate   = float(dml.ate(X))
    ci    = dml.ate_interval(X, alpha=0.05)

    return {
        "estimator": "LinearDML",
        "ate":       round(ate, 4),
        "ate_lower": round(float(ci[0]), 4),
        "ate_upper": round(float(ci[1]), 4),
        "note": "DML with Ridge nuisance; 5-fold CV; 95% CI",
        "_model": dml,
    }


# ──────────────────────────────────────────────────────────────
# Estimator 4 — Causal Forest DML (heterogeneous CATE)
# ──────────────────────────────────────────────────────────────
def run_causal_forest(arrays: dict) -> dict:
    """
    CausalForestDML from econml:
      - Nuisance: Ridge for Y~X and W~X
      - Final: Causal Forest for τ(X)
      - Returns per-observation CATE estimates
      - ATE = mean(CATE)
    """
    Y, W, X = arrays["Y"], arrays["W"], arrays["X"]

    cf = CausalForestDML(
        model_y=Ridge(alpha=1.0),
        model_t=Ridge(alpha=1.0),
        n_estimators=500,
        min_samples_leaf=3,
        max_depth=4,
        cv=5,
        random_state=42,
        discrete_treatment=False,
    )
    cf.fit(Y, W, X=X)

    cate      = cf.effect(X).ravel()
    ate       = float(np.mean(cate))
    ci        = cf.ate_interval(X, alpha=0.05)

    return {
        "estimator": "CausalForestDML",
        "ate":       round(ate, 4),
        "ate_lower": round(float(ci[0]), 4),
        "ate_upper": round(float(ci[1]), 4),
        "note": "CausalForest CATE; 5-fold CV; 95% CI",
        "_model": cf,
        "_cate": cate,
    }


# ──────────────────────────────────────────────────────────────
# Subgroup ATEs
# ──────────────────────────────────────────────────────────────
def subgroup_ates(
    cf_result: dict,
    arrays:    dict,
    labels:    pd.DataFrame,
) -> pd.DataFrame:
    """
    Compute mean CATE and bootstrap 95% CI for each subgroup:
      - By trajectory class (EE/AE/IR/DL)
      - By GPI group (favoured/unfavoured)
    """
    cate      = cf_result["_cate"]
    label_arr = arrays["label_arr"]
    favoured  = arrays["favoured"]
    rng       = np.random.default_rng(42)

    records = []

    def _ci(vals, n_boot=500):
        if len(vals) < 3:
            return np.nan, np.nan
        boots = [np.mean(rng.choice(vals, size=len(vals), replace=True))
                 for _ in range(n_boot)]
        return float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))

    # Trajectory class subgroups
    for cls in ["EE", "AE", "IR", "DL"]:
        mask = label_arr == cls
        if mask.sum() == 0:
            continue
        sub   = cate[mask]
        lo, hi = _ci(sub)
        records.append({
            "subgroup":    f"Trajectory: {cls}",
            "n":           int(mask.sum()),
            "mean_cate":   round(float(np.mean(sub)), 4),
            "cate_lower":  round(lo, 4) if not np.isnan(lo) else np.nan,
            "cate_upper":  round(hi, 4) if not np.isnan(hi) else np.nan,
        })

    # GPI subgroups
    for grp_name, mask in [
        ("GPI: favoured",   favoured),
        ("GPI: unfavoured", ~favoured),
    ]:
        sub    = cate[mask]
        lo, hi = _ci(sub)
        records.append({
            "subgroup":   grp_name,
            "n":          int(mask.sum()),
            "mean_cate":  round(float(np.mean(sub)), 4),
            "cate_lower": round(lo, 4) if not np.isnan(lo) else np.nan,
            "cate_upper": round(hi, 4) if not np.isnan(hi) else np.nan,
        })

    return pd.DataFrame(records)


# ──────────────────────────────────────────────────────────────
# Figures
# ──────────────────────────────────────────────────────────────
def plot_ate_comparison(results: list[dict]) -> None:
    """Forest plot: ATE ± 95% CI per estimator."""
    fig, ax = plt.subplots(figsize=(9, 5))

    colours = {
        "NaiveOLS":       "#888888",
        "FixedEffects":   "#4393c3",
        "LinearDML":      "#74c476",
        "CausalForestDML":"#d73027",
    }

    y_pos = list(range(len(results)))

    for i, res in enumerate(results):
        name  = res["estimator"]
        ate   = res["ate"]
        lo    = res.get("ate_lower")
        hi    = res.get("ate_upper")
        col   = colours.get(name, "grey")

        ax.scatter(ate, i, color=col, s=120, zorder=4,
                   edgecolors="white", linewidths=0.8)

        if pd.notna(lo) and pd.notna(hi):
            ax.plot([lo, hi], [i, i], color=col, lw=2.5,
                    solid_capstyle="round", zorder=3)
            ax.plot([lo, lo], [i-0.1, i+0.1], color=col, lw=2.0)
            ax.plot([hi, hi], [i-0.1, i+0.1], color=col, lw=2.0)

        ax.text(
            max(r.get("ate_upper", r["ate"]) for r in results
                if pd.notna(r.get("ate_upper", np.nan))) + 0.05,
            i,
            f"{ate:+.3f} pp",
            va="center", ha="left", fontsize=9, color=col,
        )

    ax.axvline(0, color="#333333", lw=1.0, ls="--", alpha=0.5)
    ax.set_yticks(y_pos)
    ax.set_yticklabels([r["estimator"] for r in results], fontsize=10)
    ax.set_xlabel(
        "ATE: estimated effect of +1 pp-GDP education spend\n"
        "on lower-secondary completion rate (pp)",
        fontsize=10,
    )
    ax.set_title(
        "Task E — ATE estimates with 95% CI\n"
        "(positive = spending improves completion)",
        fontsize=11, fontweight="bold",
    )
    ax.grid(axis="x", alpha=0.2, lw=0.6)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    out = FIGDIR / f"taskE_ate_comparison_{PERIOD_TAG}.png"
    fig.savefig(out, dpi=180, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  [OK] {out.name}")


def plot_cate_distribution(
    cf_result: dict,
    arrays:    dict,
) -> None:
    """Histogram + rug plot of per-observation CATE values."""
    cate = cf_result["_cate"]
    fig, ax = plt.subplots(figsize=(9, 5))

    ax.hist(cate, bins=20, color="#d73027", alpha=0.7,
            edgecolor="white", density=True)
    ax.axvline(np.mean(cate), color="#333333", lw=1.5, ls="--",
               label=f"ATE = {np.mean(cate):+.3f} pp")
    ax.axvline(0, color="#888888", lw=1.0, ls=":",
               label="Zero effect")

    # Rug for individual observations
    ax.plot(cate, np.zeros_like(cate) - 0.002,
            "|", color="#d73027", alpha=0.5, markersize=8)

    ax.set_xlabel(
        "CATE τ(X): effect of +1 pp-GDP spend on completion (pp)",
        fontsize=10,
    )
    ax.set_ylabel("Density", fontsize=10)
    ax.set_title(
        "Task E — CATE distribution (Causal Forest DML)\n"
        f"n={len(cate)} country-year observations",
        fontsize=11, fontweight="bold",
    )
    ax.legend(fontsize=9, frameon=True, edgecolor="#cccccc")
    ax.grid(alpha=0.2, lw=0.5)
    ax.spines[["top","right"]].set_visible(False)
    fig.tight_layout()
    out = FIGDIR / f"taskE_cate_distribution_{PERIOD_TAG}.png"
    fig.savefig(out, dpi=180, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  [OK] {out.name}")


def plot_subgroup_ate(sg_df: pd.DataFrame) -> None:
    """Forest plot of subgroup mean CATEs."""
    fig, ax = plt.subplots(figsize=(9, 6))

    for i, (_, row) in enumerate(sg_df.iterrows()):
        subgrp = row["subgroup"]
        mc     = row["mean_cate"]
        lo     = row["cate_lower"]
        hi     = row["cate_upper"]

        # Colour by subgroup type
        if "EE" in subgrp:   col = CLASS_COLOURS["EE"]
        elif "AE" in subgrp: col = CLASS_COLOURS["AE"]
        elif "IR" in subgrp: col = CLASS_COLOURS["IR"]
        elif "DL" in subgrp: col = CLASS_COLOURS["DL"]
        elif "favour" in subgrp.lower():
            col = "#4393c3" if "unfav" not in subgrp.lower() else "#d6604d"
        else:
            col = "#555555"

        ax.scatter(mc, i, color=col, s=120, zorder=4,
                   edgecolors="white", linewidths=0.8)
        if pd.notna(lo) and pd.notna(hi):
            ax.plot([lo, hi], [i, i], color=col, lw=2.5,
                    solid_capstyle="round")
            for x in [lo, hi]:
                ax.plot([x, x], [i-0.1, i+0.1], color=col, lw=1.8)

        ax.text(
            sg_df[["cate_upper","mean_cate"]].apply(
                lambda r: r["cate_upper"] if pd.notna(r["cate_upper"])
                else r["mean_cate"], axis=1).max() + 0.05,
            i,
            f"{mc:+.3f} pp  (n={int(row['n'])})",
            va="center", ha="left", fontsize=8.5, color=col,
        )

    ax.axvline(0, color="#333333", lw=1.0, ls="--", alpha=0.5)
    ax.set_yticks(range(len(sg_df)))
    ax.set_yticklabels(sg_df["subgroup"], fontsize=9)
    ax.set_xlabel(
        "Mean CATE: effect of +1 pp-GDP spend on completion (pp)",
        fontsize=10,
    )
    ax.set_title(
        "Task E — Subgroup CATE estimates (Causal Forest DML)\n"
        "Bootstrap 95% CI; subgroups: trajectory class and GPI group",
        fontsize=10, fontweight="bold",
    )
    ax.grid(axis="x", alpha=0.2, lw=0.6)
    ax.spines[["top","right"]].set_visible(False)
    fig.tight_layout()
    out = FIGDIR / f"taskE_subgroup_ate_{PERIOD_TAG}.png"
    fig.savefig(out, dpi=180, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  [OK] {out.name}")


# ──────────────────────────────────────────────────────────────
# Save outputs
# ──────────────────────────────────────────────────────────────
def save_results(
    results:    list[dict],
    sg_df:      pd.DataFrame,
    cf_result:  dict,
    arrays:     dict,
) -> None:
    print("\n[step 5] Saving results …")

    # ATE table
    ate_df = pd.DataFrame([
        {k: v for k, v in r.items() if not k.startswith("_")}
        for r in results
    ])
    ate_df.to_csv(RESULT_DIR / "task_E_ate.csv",    index=False)
    ate_df.to_csv(TABLE_DIR  / "table_taskE_ate.csv", index=False)
    print(f"  [OK] task_E_ate.csv")

    # CATE per observation
    cate_df = arrays["df"][["iso3c","country","income_group","year"]].copy()
    cate_df["cate"] = cf_result["_cate"]
    cate_df["label"] = arrays["label_arr"]
    cate_df.to_csv(RESULT_DIR / "task_E_cate.csv", index=False)
    print(f"  [OK] task_E_cate.csv  ({len(cate_df)} rows)")

    # Subgroup CATE
    sg_df.to_csv(TABLE_DIR / "table_taskE_subgroup.csv", index=False)

    # LaTeX
    lines = [
        r"\begin{table}[htbp]",
        r"\centering",
        r"\caption{Task E --- ATE estimates: effect of +1 pp-GDP "
        r"education expenditure on lower-secondary completion rate (pp). "
        r"95\% CI reported; $n=" + str(len(arrays['Y'])) + r"$ "
        r"country-year observations (2016--2024, imputed panel).}",
        r"\label{tab:taskE}",
        r"\begin{tabular}{lrrrp{5cm}}",
        r"\toprule",
        r"Estimator & ATE & CI$_{2.5}$ & CI$_{97.5}$ & Note \\",
        r"\midrule",
    ]
    def f(v): return f"{v:+.3f}" if pd.notna(v) else "--"
    for _, row in ate_df.iterrows():
        name = row["estimator"].replace("_", r"\_")
        lines.append(
            f"{name} & {f(row['ate'])} & "
            f"{f(row.get('ate_lower', np.nan))} & "
            f"{f(row.get('ate_upper', np.nan))} & "
            f"\\footnotesize{{{row.get('note','')}}} \\\\"
        )
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    (TABLE_DIR / "table_taskE.tex").write_text(
        "\n".join(lines), encoding="utf-8"
    )
    print(f"  [OK] table_taskE.csv / .tex")
    print(f"\n[done] Results : {RESULT_DIR.resolve()}")
    print(f"[done] Tables  : {TABLE_DIR.resolve()}")
    print(f"[done] Figures : {FIGDIR.resolve()}")
    print(f"\n{'='*60}")
    print("Pipeline complete — all 5 tasks run successfully.")
    print(f"{'='*60}")
    print("\nOutputs summary:")
    print(f"  data/processed/                 panel + masks + labels + splits")
    print(f"  outputs/results/                task_[A-E]_metrics/predictions")
    print(f"  outputs/tables/                 table_[1-3] + table_task[A-E]")
    print(f"  outputs/figures/panel_12/       all figures for manuscript")


def print_summary(results: list[dict], sg_df: pd.DataFrame) -> None:
    print(f"\n{'='*65}")
    print("Task E — Causal ML results")
    print(f"{'='*65}")
    print(f"  {'Estimator':<20}  {'ATE':>8}  {'CI_lo':>8}  {'CI_hi':>8}")
    print(f"  {'-'*20}  {'-'*8}  {'-'*8}  {'-'*8}")
    for r in results:
        def f(v, w=8): return f"{v:+{w}.3f}" if pd.notna(v) else f"{'--':>{w}}"
        print(f"  {r['estimator']:<20}  {f(r['ate'])}  "
              f"{f(r.get('ate_lower', np.nan))}  "
              f"{f(r.get('ate_upper', np.nan))}")
    print(f"\n  Subgroup CATEs (Causal Forest):")
    for _, row in sg_df.iterrows():
        def f(v, w=8): return f"{v:+{w}.3f}" if pd.notna(v) else f"{'--':>{w}}"
        print(f"    {row['subgroup']:<30}  "
              f"CATE={f(row['mean_cate'])}  n={int(row['n'])}")
    print(f"{'='*65}")


# ──────────────────────────────────────────────────────────────
# Main
# ──────────────────────────────────────────────────────────────
def main() -> None:
    panel, labels = load_data()
    arrays = build_causal_arrays(panel, labels)

    print("\n[step 2] Running causal estimators …")

    print("  [1/4] Naïve OLS …")
    ols_result = run_naive_ols(arrays)
    print(f"        ATE = {ols_result['ate']:+.3f} pp (confounded)")

    print("  [2/4] Fixed Effects (within demeaning) …")
    fe_result = run_fixed_effects(arrays)
    print(f"        ATE = {fe_result['ate']:+.3f} pp  "
          f"95% CI [{fe_result['ate_lower']:+.3f}, {fe_result['ate_upper']:+.3f}]")

    print("  [3/4] LinearDML …")
    dml_result = run_linear_dml(arrays)
    print(f"        ATE = {dml_result['ate']:+.3f} pp  "
          f"95% CI [{dml_result['ate_lower']:+.3f}, {dml_result['ate_upper']:+.3f}]")

    print("  [4/4] CausalForestDML …")
    cf_result = run_causal_forest(arrays)
    print(f"        ATE = {cf_result['ate']:+.3f} pp  "
          f"95% CI [{cf_result['ate_lower']:+.3f}, {cf_result['ate_upper']:+.3f}]")

    all_results = [ols_result, fe_result, dml_result, cf_result]

    print("\n[step 3] Subgroup CATEs …")
    sg_df = subgroup_ates(cf_result, arrays, labels)

    print_summary(all_results, sg_df)

    print("\n[step 4] Figures …")
    plot_ate_comparison(all_results)
    plot_cate_distribution(cf_result, arrays)
    plot_subgroup_ate(sg_df)

    save_results(all_results, sg_df, cf_result, arrays)


if __name__ == "__main__":
    main()